# HealthBot: AI-Powered Patient Education System

A LangGraph-based workflow that provides personalized health information to patients,
summarizes medical topics, and assesses understanding through quizzes.

## 1. Environment Setup

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv('config.env', override=True)

_openai_key = os.getenv('OPENAI_API_KEY')
_tavily_key = os.getenv('TAVILY_API_KEY')

assert _openai_key is not None, "OPENAI_API_KEY is missing in config.env"
assert _tavily_key is not None, "TAVILY_API_KEY is missing in config.env"

print("API keys loaded successfully")

In [ ]:
from typing import TypedDict, Literal, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
import textwrap

print("All imports successful")

## 2. Initialize & Verify Tavily Tool

Creates the Tavily search tool and immediately runs a test query to verify it works.
The results are printed so you can visually confirm Tavily is responding correctly.

In [ ]:
tavily_tool = TavilySearchResults(
    max_results=5,
    include_answer=True,
    include_raw_content=True,
)

print("=" * 70)
print("  TAVILY TOOL VERIFICATION")
print("=" * 70)
print(f"Created: {tavily_tool.name}\n")

print("Running default test search: 'heart disease symptoms treatment'...")
print()

try:
    test_result = tavily_tool.invoke({"query": "heart disease symptoms treatment"})

    if test_result and len(test_result) > 0:
        print(f"PASS: Got {len(test_result)} results back from Tavily.\n")
        print("--- Results Preview ---")
        for i, item in enumerate(test_result[:3], 1):
            if isinstance(item, dict):
                title = item.get('title', 'N/A')
                url = item.get('url', '')
                content = item.get('content', '')
                print(f"  [{i}] {title}")
                print(f"      URL: {url}")
                wrapped = textwrap.fill(content[:300], width=66,
                                        initial_indent='      ', subsequent_indent='      ')
                print(wrapped)
                print()
        print("--- End of Preview ---")
        TAVILY_OK = True
    else:
        print("WARNING: Tavily returned empty results.")
        print("Check your TAVILY_API_KEY in config.env")
        TAVILY_OK = False
except Exception as e:
    print(f"FAILED: {e}")
    TAVILY_OK = False

print("=" * 70)

tools = [tavily_tool]
tool_executor = ToolNode(tools)

## 3. Define State

In [ ]:
class HealthBotState(TypedDict):
    messages: Annotated[list, add_messages]
    topic: str
    search_results: str
    summary: str
    search_successful: bool
    quiz_question: str
    user_answer: str
    grade_feedback: str
    waiting_for_input: str

print("State class defined")

## 4. Define LLM

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.3,
)

llm_with_tools = llm.bind_tools(tools)
print("LLM configured with Tavily tool binding")

## 5. Define Workflow Nodes

In [ ]:
AFFIRMATIVE = {"yes", "y", "yeah", "yep", "sure", "ok", "okay", "ready", "r"}
NEGATIVE = {"no", "n", "nope", "nah", "exit", "quit", "stop", "end"}


def normalize_choice(text: str) -> str:
    t = text.strip().lower()
    if t in AFFIRMATIVE:
        return "yes"
    if t in NEGATIVE:
        return "no"
    return t


def ask_topic(state: HealthBotState) -> dict:
    topic = input("\nWhat health topic or medical condition would you like to learn about today? ").strip()
    while not topic:
        print("Please enter a valid health topic.")
        topic = input("What health topic or medical condition would you like to learn about today? ").strip()
    print(f"\nSearching for reliable information about '{topic}'...")
    return {
        "topic": topic,
        "search_results": "",
        "summary": "",
        "search_successful": False,
        "quiz_question": "",
        "user_answer": "",
        "grade_feedback": "",
        "waiting_for_input": "topic_received",
        "messages": [HumanMessage(content=f"I want to learn about: {topic}")],
    }


def search_tavily(state: HealthBotState) -> dict:
    topic = state["topic"]
    system_prompt = SystemMessage(
        content=(
            f"You are a medical research assistant. Your ONLY job is to call the "
            f"tavily_search_results_json tool to search for '{topic}'.\n\n"
            f"You MUST call the tool. Do NOT answer with text. "
            f"After the tool returns, simply say 'done'."
        )
    )
    response = llm_with_tools.invoke([system_prompt])
    return {"messages": [response]}


def execute_tools(state: HealthBotState) -> dict:
    messages = state.get("messages", [])
    last_message = messages[-1] if messages else None

    if not last_message or not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
        print("  [No tool calls found in last message]")
        return {"search_results": "", "search_successful": False}

    search_text_parts = []
    for tool_call in last_message.tool_calls:
        args = tool_call.get("args", {}) if isinstance(tool_call, dict) else tool_call.args
        try:
            result = tavily_tool.invoke(args)
            search_text_parts.append(str(result))
        except Exception as e:
            print(f"  [Tool error: {e}]")
            search_text_parts.append(f"Error: {e}")

    combined = "\n\n---\n\n".join(search_text_parts)
    successful = len(combined.strip()) >= 50

    if not successful:
        print(f"  [Search returned insufficient content ({len(combined.strip())} chars)]")

    return {
        "search_results": combined,
        "search_successful": successful,
    }


def route_after_search(state: HealthBotState) -> Literal["execute_tools", "search_tavily", "summarize_results"]:
    messages = state.get("messages", [])
    if messages and hasattr(messages[-1], "tool_calls") and messages[-1].tool_calls:
        return "execute_tools"
    print("  [Retrying - LLM did not call search tool]")
    return "search_tavily"


SEARCH_FAILURE_PHRASES = [
    "unable to retrieve", "no search results", "no results",
    "could not find", "i'm sorry", "i cannot", "no information",
    "was unable", "not able to", "no data", "nothing found",
]


def is_failure_summary(text: str) -> bool:
    t = text.lower()
    return any(p in t for p in SEARCH_FAILURE_PHRASES)


def summarize_results(state: HealthBotState) -> dict:
    search_text = state.get("search_results", "")
    if not state.get("search_successful", False):
        fail_msg = (
            "I'm sorry, but I was unable to retrieve search results for this topic. "
            "Please try a different health topic."
        )
        return {
            "summary": fail_msg,
            "search_successful": False,
            "messages": [AIMessage(content=fail_msg)],
        }
    system_prompt = SystemMessage(
        content=(
            "You are a patient education specialist. Summarize the search results below "
            "into 3-4 paragraphs of plain, simple language.\n"
            "Rules: Use ONLY the provided information. No external knowledge. "
            "Be warm and encouraging.\n\n"
            f"Search results:\n\n{search_text}"
        )
    )
    response = llm.invoke([system_prompt])
    summary_text = response.content
    actually_successful = not is_failure_summary(summary_text) and len(summary_text.strip()) >= 50
    return {
        "summary": summary_text,
        "search_successful": actually_successful,
        "messages": [AIMessage(content=summary_text)],
    }


def present_summary(state: HealthBotState) -> None:
    print("\n" + "=" * 70)
    print("HEALTH INFORMATION SUMMARY")
    print("=" * 70)
    print(state["summary"])
    print("=" * 70 + "\n")


def route_after_summary(state: HealthBotState) -> Literal["ask_ready_for_quiz", "ask_retry_or_exit"]:
    if state.get("search_successful", False):
        return "ask_ready_for_quiz"
    return "ask_retry_or_exit"


def ask_retry_or_exit(state: HealthBotState) -> dict:
    print("The search for this topic did not return usable results.")
    choice = normalize_choice(input("Would you like to try a different topic? (yes/no): "))
    while choice not in ["yes", "no"]:
        print("Please answer 'yes' or 'no'.")
        choice = normalize_choice(input("Try a different topic? (yes/no): "))
    return {"waiting_for_input": choice}


def route_retry_or_exit(state: HealthBotState) -> Literal["reset_state", "end_session"]:
    if state.get("waiting_for_input", "") == "yes":
        return "reset_state"
    return "end_session"


def ask_ready_for_quiz(state: HealthBotState) -> dict:
    print("Now let's check your understanding with a quick question.")
    resp = normalize_choice(input("Type 'ready' when you are ready for the quiz: "))
    attempts = 0
    while resp != "yes" and attempts < 5:
        if resp == "no":
            print("Take your time. Let me know when you are ready.")
        else:
            print("Please type 'ready' (or 'yes') when you want to proceed.")
        resp = normalize_choice(input("Ready for the quiz? "))
        attempts += 1
    return {"waiting_for_input": "quiz_ready"}


def generate_quiz(state: HealthBotState) -> dict:
    system_prompt = SystemMessage(
        content=(
            "You are a patient education specialist creating a comprehension check.\n\n"
            "Create exactly ONE quiz question based ONLY on the summary below.\n"
            "The question must be answerable using ONLY the summary.\n"
            "Test a key point from the information.\n"
            "Output just the question, nothing else.\n\n"
            f"Summary:\n\n{state['summary']}"
        )
    )
    response = llm.invoke([system_prompt])
    return {
        "quiz_question": response.content,
        "messages": [AIMessage(content=f"Quiz question: {response.content}")],
    }


def present_quiz(state: HealthBotState) -> None:
    print("\n" + "-" * 70)
    print("COMPREHENSION CHECK")
    print("-" * 70)
    print(state["quiz_question"])
    print("-" * 70 + "\n")


def get_answer(state: HealthBotState) -> dict:
    print("Please answer based on what you learned from the summary.")
    answer = input("Your answer: ").strip()
    while not answer:
        print("Please enter an answer.")
        answer = input("Your answer: ").strip()
    return {
        "user_answer": answer,
        "waiting_for_input": "answer_received",
    }


def grade_answer(state: HealthBotState) -> dict:
    system_prompt = SystemMessage(
        content=(
            "You are grading a comprehension check. Use ONLY the summary as reference.\n\n"
            "Provide:\n"
            "1. A letter grade (A, B, C, D, or F)\n"
            "2. A justification for the grade\n"
            "3. Citations from the summary to reinforce learning\n"
            "\nBe encouraging and educational.\n\n"
            f"Summary:\n{state['summary']}\n\n"
            f"Question:\n{state['quiz_question']}\n\n"
            f"Answer:\n{state['user_answer']}"
        )
    )
    response = llm.invoke([system_prompt])
    return {
        "grade_feedback": response.content,
        "messages": [AIMessage(content=response.content)],
    }


def present_grade(state: HealthBotState) -> None:
    print("\n" + "=" * 70)
    print("GRADE AND FEEDBACK")
    print("=" * 70)
    print(state["grade_feedback"])
    print("=" * 70 + "\n")


def ask_continue(state: HealthBotState) -> dict:
    print("Thank you for using HealthBot today!")
    choice = normalize_choice(input("Would you like to learn about another health topic? (yes/no): "))
    while choice not in ["yes", "no"]:
        print("Please answer 'yes' or 'no'.")
        choice = normalize_choice(input("Learn about another topic? (yes/no): "))
    return {"waiting_for_input": choice}


def route_after_continue(state: HealthBotState) -> Literal["reset_state", "end_session"]:
    if state.get("waiting_for_input", "") == "yes":
        return "reset_state"
    return "end_session"


def reset_state(state: HealthBotState) -> dict:
    print("\n" + "-" * 70)
    print("Starting a new topic. Previous information cleared.")
    print("-" * 70 + "\n")
    return {
        "topic": "",
        "search_results": "",
        "summary": "",
        "search_successful": False,
        "quiz_question": "",
        "user_answer": "",
        "grade_feedback": "",
        "waiting_for_input": "new_topic",
    }


def end_session(state: HealthBotState) -> None:
    print("\n" + "=" * 70)
    print("Thank you for using HealthBot! Take care of your health.")
    print("=" * 70)


print("All node functions defined")

## 6. Build the Graph

In [ ]:
builder = StateGraph(HealthBotState)

builder.add_node("ask_topic", ask_topic)
builder.add_node("search_tavily", search_tavily)
builder.add_node("execute_tools", execute_tools)
builder.add_node("summarize_results", summarize_results)
builder.add_node("present_summary", present_summary)
builder.add_node("ask_retry_or_exit", ask_retry_or_exit)
builder.add_node("ask_ready_for_quiz", ask_ready_for_quiz)
builder.add_node("generate_quiz", generate_quiz)
builder.add_node("present_quiz", present_quiz)
builder.add_node("get_answer", get_answer)
builder.add_node("grade_answer", grade_answer)
builder.add_node("present_grade", present_grade)
builder.add_node("ask_continue", ask_continue)
builder.add_node("reset_state", reset_state)
builder.add_node("end_session", end_session)

builder.add_edge(START, "ask_topic")
builder.add_edge("ask_topic", "search_tavily")

# If LLM calls tool -> execute. If not -> retry search (loops until tool is called)
builder.add_conditional_edges(
    "search_tavily",
    route_after_search,
    {"execute_tools": "execute_tools", "search_tavily": "search_tavily", "summarize_results": "summarize_results"},
)

builder.add_edge("execute_tools", "summarize_results")
builder.add_edge("summarize_results", "present_summary")

# After presenting: success -> quiz, failure -> retry/exit
builder.add_conditional_edges(
    "present_summary",
    route_after_summary,
    {"ask_ready_for_quiz": "ask_ready_for_quiz", "ask_retry_or_exit": "ask_retry_or_exit"},
)

builder.add_conditional_edges(
    "ask_retry_or_exit",
    route_retry_or_exit,
    {"reset_state": "reset_state", "end_session": "end_session"},
)

builder.add_edge("ask_ready_for_quiz", "generate_quiz")
builder.add_edge("generate_quiz", "present_quiz")
builder.add_edge("present_quiz", "get_answer")
builder.add_edge("get_answer", "grade_answer")
builder.add_edge("grade_answer", "present_grade")
builder.add_edge("present_grade", "ask_continue")

builder.add_conditional_edges(
    "ask_continue",
    route_after_continue,
    {"reset_state": "reset_state", "end_session": "end_session"},
)

builder.add_edge("reset_state", "ask_topic")
builder.add_edge("end_session", END)

graph = builder.compile()
print("LangGraph workflow built and compiled successfully")

## 7. Visualize the Graph

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("Graph visualization not available.")

## 8. Run the HealthBot

In [ ]:
if not TAVILY_OK:
    print("ERROR: Tavily tool is not responding. Fix the API key issue (see Section 2 output)")
    print("then restart this notebook (Kernel > Restart & Run All).")
else:
    print("=" * 70)
    print("     Welcome to HealthBot - AI-Powered Patient Education")
    print("=" * 70)

    initial_state: HealthBotState = {
        "messages": [],
        "topic": "",
        "search_results": "",
        "summary": "",
        "search_successful": False,
        "quiz_question": "",
        "user_answer": "",
        "grade_feedback": "",
        "waiting_for_input": "",
    }

    for event in graph.stream(initial_state):
        pass

    print("\nHealthBot session ended.")